# 03 — Live-feed streaming integration

Start a `stream.Server` on localhost, push frames from a `stream.Client`,
get 2D cakes back over the socket. v0.1.0 is CPU-backed and single-client;
add `backend='gpu'` once the GPU wheel is installed to hit the paper's
1700+ FPS numbers on H100.

In [1]:
import socket, tempfile
from pathlib import Path

import numpy as np
import tifffile

import midas_auto_calibrate as mac
import midas_integrate as mi

## Setup — pre-build the detector map

Mapper output is reused across every frame the server processes.

In [2]:
workdir = Path(tempfile.mkdtemp(prefix='mi_stream_'))
cfg = mi.IntegrationConfig(
    lsd=657_436.9, ybc=685.5, zbc=921.0,
    wavelength=0.172973, pixel_size=172.0,
    nr_pixels_y=1475, nr_pixels_z=1679,
    r_min=50, r_max=1000, r_bin_size=1.0,
    eta_min=-180, eta_max=180, eta_bin_size=4.0,
)
artefacts = mi.Mapper(cfg).build(workdir, n_cpus=4)

## Grab a free TCP port (avoid collisions with other beamline daemons)

In [3]:
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(('127.0.0.1', 0))
    port = s.getsockname()[1]
print(f'will bind server to 127.0.0.1:{port}')

will bind server to 127.0.0.1:59840


## Start the server + push frames

`Server` runs in a background thread; the `with` block joins on exit.

In [4]:
frame = tifffile.imread(mac.data.CEO2_PILATUS).astype(np.uint32)

with mi.stream.Server(cfg, artefacts, port=port) as srv:
    with mi.stream.Client(f'127.0.0.1:{port}', op_timeout=120.0) as c:
        # Send the same frame three times to simulate a scan.
        cakes = [c.send_frame(frame) for _ in range(3)]

print(f'received {len(cakes)} cakes, each shape {cakes[0].shape}')
print(f'max intensity across cakes: {[c.max() for c in cakes]}')

received 3 cakes, each shape (950, 90)
max intensity across cakes: [np.float32(4.6917325e+09), np.float32(4.6917325e+09), np.float32(4.6917325e+09)]


## Health-check: round-trip PING

In [5]:
# Launch a new server for the ping demo.
with mi.stream.Server(cfg, artefacts, port=port) as srv:
    with mi.stream.Client(f'127.0.0.1:{port}') as c:
        rtt_ms = c.ping() * 1000
print(f'PING round-trip: {rtt_ms:.2f} ms')

PING round-trip: 27.66 ms


## Command-line alternative

```bash
# In one terminal — start the server
midas-integrate-server Parameters.txt --port 60439

# In another — push frames, save cakes as .npy
midas-integrate-client 127.0.0.1:60439 frame_*.tif --out ./cakes/
```